# Cross-domain and cross-database evaluation
**Inputs:** 12 final model bundles, their metadata and six target split tables. **Outputs:** 72 source-model/target cases under two separately labelled protocols (144 metric rows), overlap counts, predictions, per-element summaries and a versioned state file under `COD_CANON_V2/results/cross/PAPER_V2_CROSS/`.

`all_test` uses the complete target test set. `composition_disjoint_test` removes target test records sharing a composition group with source training records. Cross-project composition matching uses exact rational keys followed by atomic-fraction tolerance 1e-5 and connected components. No model is retrained. Both protocols use C=0/N=1 and threshold 0.5.

Inference only. Each source model is one final job from `03_train_models.ipynb`, selected by its own `meta_*.json`. Each task is that model scored on one test domain.

`FORCE = False` reuses a finished task in the current experiment version when the model, folds, features, class mapping, threshold, composition rule and report settings still match. A mismatch stops the run. An interrupted task is repeated. `FORCE = True` starts a new version and keeps the previous results. The next `FORCE = False` resumes that new version.

Select metadata explicitly through `RUN_OVERRIDES` if multiple final runs exist. Hash checks cover models, splits, features and configuration; a changed experiment requires a new version. The original reported cross-check is `v0001`.


In [ ]:
from pathlib import Path
import sys

REPO_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "project_config.py").is_file()), None)
if REPO_ROOT is None:
    raise RuntimeError("Open this notebook from the NLO_ML repository.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

FORCE = False
EXPERIMENT_ID = "PAPER_V2_CROSS"
PAPER_RUN = "PAPER_V2"
PROTOCOL_VERSION = 1
OVERLAP_METHOD = "exact_rational_then_l_infinity_connected_components"
COMPOSITION_TOLERANCE = 1e-5
SAVE_PREDICTIONS = True
SAVE_PER_ELEMENT = True
ELEMENT_MIN_N = 30

from project_config import make_cfg
from private_paths import COD_DIR, MP_DIR
from reports.paper_inputs import selected_runs
RUN_OVERRIDES = {}
configs = [make_cfg(project_name="COD_CANON_V2", root_cif=COD_DIR),
           make_cfg(project_name="MP_ALL", root_cif=MP_DIR)]
SOURCE_MODELS = [
    {"project": "COD" if cfg.PROJECT_NAME == "COD_CANON_V2" else "MP",
     "model": model, "domain": domain, "manifest": f"meta_{run}.json"}
    for cfg, model, domain, run in selected_runs(configs, paper_run=PAPER_RUN,
                                               overrides=RUN_OVERRIDES)
]
TEST_TARGETS = (
    ("COD", "N"), ("COD", "O"), ("COD", "M"),
    ("MP", "N"), ("MP", "O"), ("MP", "M"),
)


## Define the evaluation routines
The following cell defines the validation, evaluation and checkpoint routines. The final execution cell runs the configured cases.


In [ ]:
import json
import time
import traceback
from importlib.metadata import PackageNotFoundError, version

import numpy as np
import pandas as pd
from tqdm import tqdm

from core import ml_utils as mlu
from core.data_preparation_csv import group_by_composition_and_fix_labels
from core.notebook_logger import NotebookLogger

PROJECTS = {
    "COD": make_cfg(project_name="COD_CANON_V2", root_cif=COD_DIR),
    "MP": make_cfg(project_name="MP_ALL", root_cif=MP_DIR),
}

def _config_fingerprint():
    return mlu.json_fingerprint({
        "experiment_id": EXPERIMENT_ID,
        "protocol_version": PROTOCOL_VERSION,
        "overlap_method": OVERLAP_METHOD,
        "composition_tolerance": COMPOSITION_TOLERANCE,
        "save_predictions": SAVE_PREDICTIONS,
        "save_per_element": SAVE_PER_ELEMENT,
        "element_min_n": ELEMENT_MIN_N,
    })

def _task_fingerprint(identity):
    payload = {
        "experiment_id": EXPERIMENT_ID,
        "protocol_version": PROTOCOL_VERSION,
        "overlap_method": OVERLAP_METHOD,
        "composition_tolerance": COMPOSITION_TOLERANCE,
        "save_predictions": SAVE_PREDICTIONS,
        "save_per_element": SAVE_PER_ELEMENT,
        "element_min_n": ELEMENT_MIN_N,
    }
    payload.update(identity)
    return mlu.json_fingerprint(payload)

def _atomic_json(path, payload):
    mlu.atomic_text(path, json.dumps(payload, indent=2, default=str))

def _atomic_csv(path, frame):
    mlu.atomic_text(path, frame.to_csv(index=False))

def _experiment_dir():
    return Path(PROJECTS["COD"].RESULTS_DIR) / "cross" / EXPERIMENT_ID

def _version_name(index):
    return f"v{index:04d}"

def _open_version():
    root = _experiment_dir()
    root.mkdir(parents=True, exist_ok=True)
    pointer = root / "current.json"
    if FORCE:
        previous = 0
        if pointer.is_file():
            recorded = json.loads(pointer.read_text(encoding="utf-8"))
            if recorded.get("experiment_id") != EXPERIMENT_ID:
                raise RuntimeError(f"current.json belongs to another experiment: {pointer}")
            previous = int(str(recorded["version"]).lstrip("v"))
        version = _version_name(previous + 1)
        version_dir = root / version
        version_dir.mkdir(parents=False, exist_ok=False)
        _atomic_json(pointer, {"experiment_id": EXPERIMENT_ID, "version": version})
        state = {
            "experiment_id": EXPERIMENT_ID,
            "version": version,
            "config_fingerprint": _config_fingerprint(),
            "status": "running",
            "tasks": {},
        }
        _atomic_json(version_dir / "state.json", state)
        print(f"[FORCE] New version {version}. Previous results are kept.")
        return version_dir, state
    if not pointer.is_file():
        version = _version_name(1)
        version_dir = root / version
        version_dir.mkdir(parents=False, exist_ok=False)
        state = {
            "experiment_id": EXPERIMENT_ID,
            "version": version,
            "config_fingerprint": _config_fingerprint(),
            "status": "running",
            "tasks": {},
        }
        _atomic_json(version_dir / "state.json", state)
        _atomic_json(pointer, {"experiment_id": EXPERIMENT_ID, "version": version})
        return version_dir, state
    recorded = json.loads(pointer.read_text(encoding="utf-8"))
    if recorded.get("experiment_id") != EXPERIMENT_ID:
        raise RuntimeError(f"current.json belongs to another experiment: {pointer}")
    version_dir = root / recorded["version"]
    state_path = version_dir / "state.json"
    if not state_path.is_file():
        raise RuntimeError(f"Missing state for the current version: {state_path}")
    state = json.loads(state_path.read_text(encoding="utf-8"))
    if state.get("config_fingerprint") != _config_fingerprint():
        raise RuntimeError(
            "The current experiment version used a different protocol, composition "
            "tolerance, overlap method or report settings. Set FORCE=True to start "
            f"a new version. State: {state_path}"
        )
    print(f"Resuming {recorded['version']} at {version_dir}")
    return version_dir, state

def _save_state(version_dir, state):
    _atomic_json(version_dir / "state.json", state)

def _require_sources():
    missing = [
        f"{item['project']} {item['model']} {item['domain']}"
        for item in SOURCE_MODELS if not item.get("manifest")
    ]
    if missing:
        raise RuntimeError(
            "Set manifest to the meta JSON of each selected 03 final job. "
            "Missing: " + ", ".join(missing)
        )
    seen = set()
    for item in SOURCE_MODELS:
        key = (item["project"], item["model"], item["domain"])
        if key in seen:
            raise RuntimeError(f"Duplicate source selection: {key}")
        seen.add(key)
        if item["project"] not in PROJECTS:
            raise RuntimeError(f"Unknown project {item['project']}")

def _load_source(item):
    cfg = PROJECTS[item["project"]]
    manifest_path = Path(cfg.RESULTS_DIR) / item["manifest"]
    if not manifest_path.is_file():
        raise FileNotFoundError(f"Missing 03 manifest: {manifest_path}")
    meta = json.loads(manifest_path.read_text(encoding="utf-8"))
    if (meta.get("mode"), meta.get("model"), meta.get("step"), meta.get("domain"), meta.get("project")) != (
            "final", item["model"], "magpie", item["domain"], cfg.PROJECT_NAME):
        raise RuntimeError(f"Manifest does not match the selected final job: {manifest_path}")
    run_id = meta.get("run_id")
    if not str(run_id).startswith(f"{PAPER_RUN}__{cfg.PROJECT_NAME}__final__{item['model']}__magpie__{item['domain']}__"):
        raise RuntimeError(f"Unexpected final run id in {manifest_path}: {run_id}")
    model_path = Path(cfg.MODELS_DIR) / f"model_{run_id}.pkl"
    if not model_path.is_file():
        raise FileNotFoundError(model_path)
    folds = Path(cfg.DATASET_FOLDS_TEMPLATE.format(domain=item["domain"]))
    return {
        "project": item["project"],
        "model": item["model"],
        "domain": item["domain"],
        "run_id": run_id,
        "manifest_path": manifest_path,
        "manifest_sha256": mlu.file_sha256(manifest_path),
        "model_path": model_path,
        "model_sha256": mlu.file_sha256(model_path),
        "folds_path": folds,
        "folds_sha256": mlu.file_sha256(folds),
    }

def _task_key(source, target_project, target_domain):
    return f"{source['model']}__{source['project']}_{source['domain']}__to__{target_project}_{target_domain}"

def _assess_saved(record, identity, version_dir):
    """Return reuse or redo. A finished record that disagrees with the files is an error."""
    if not record or record.get("status") != "complete":
        return "redo"
    fields = (
        "run_id", "model_sha256", "source_domain", "target_domain",
        "source_folds_sha256", "target_folds_sha256", "manifest_sha256",
        "features_sha256", "mapping", "threshold",
    )
    for field in fields:
        if record.get(field) != identity.get(field):
            raise RuntimeError(
                f"{identity['task_key']}: finished comparison {field} does not match the current "
                f"inputs. Saved {record.get(field)!r}, current {identity.get(field)!r}. "
                "It was not reused."
            )
    if record.get("fingerprint") != _task_fingerprint(identity):
        raise RuntimeError(
            f"{identity['task_key']}: finished comparison fingerprint does not match the "
            "protocol, composition rule or report settings. It was not reused."
        )
    for artifact in record.get("artifacts") or []:
        path = version_dir / artifact["name"]
        if not path.is_file() or mlu.file_sha256(path) != artifact["sha256"]:
            raise RuntimeError(
                f"Finished artifact content changed: {path}. The saved comparison was not reused."
            )
    return "reuse"

def _load_fold_data(cfg, domain):
    path = Path(cfg.DATASET_FOLDS_TEMPLATE.format(domain=domain))
    header = pd.read_csv(path, nrows=0).columns.tolist()
    features = [col for col in header if col.startswith("MagpieData ")]
    meta_cols = ["id", "chem_formula_cif", "centrosymmetry", "chem_class", "group_id", "elements_cif", "is_test"]
    if len(features) != 132 or not set(meta_cols).issubset(header):
        raise ValueError(f"Unexpected Magpie/metadata schema: {path}")
    metadata, tests = [], []
    for chunk in pd.read_csv(path, dtype={"id": str, "group_id": str}, usecols=meta_cols + features, chunksize=25000):
        flags = chunk["is_test"].astype(str).str.lower()
        if not flags.isin(["0", "1", "false", "true", "no", "yes"]).all():
            raise ValueError(f"Invalid is_test flags: {path}")
        chunk["is_test"] = flags.isin(["1", "true", "yes"])
        if (chunk[meta_cols].isna().any().any()
                or not chunk["centrosymmetry"].isin(["C", "N"]).all()
                or not chunk["chem_class"].eq(domain).all()
                or not np.isfinite(chunk[features].to_numpy(dtype=float)).all()):
            raise ValueError(f"Invalid labels, metadata or features: {path}")
        metadata.append(chunk[meta_cols].copy())
        tests.append(chunk.loc[chunk["is_test"]].copy())
    metadata = pd.concat(metadata, ignore_index=True)
    test = pd.concat(tests, ignore_index=True)
    if metadata["id"].duplicated().any() or test.empty or metadata["is_test"].all():
        raise ValueError(f"Duplicate IDs or empty train/test: {path}")
    if set(metadata.loc[~metadata.is_test, "group_id"]) & set(test["group_id"]):
        raise ValueError(f"Train/test group overlap: {path}")
    return metadata, test

def _write_tables(version_dir, state):
    metric_rows, overlap_rows = [], []
    for record in state["tasks"].values():
        if record.get("status") != "complete":
            continue
        metric_rows.extend(record["metrics"])
        overlap_rows.append(record["overlap"])
    _atomic_csv(version_dir / "cross_metrics.csv", pd.DataFrame(metric_rows))
    _atomic_csv(version_dir / "cross_overlap_summary.csv", pd.DataFrame(overlap_rows))

def _bundle_identity(bundle, source):
    spec = bundle.target_spec
    raw_mapping = spec.get("mapping") if isinstance(spec, dict) else spec.mapping
    mapping = {str(label): int(code) for label, code in dict(raw_mapping).items()}
    if (bundle.run_id != source["run_id"] or bundle.domain != source["domain"]
            or bundle.task != "classification" or mapping != {"C": 0, "N": 1}):
        raise RuntimeError(f"Bundle identity/target mismatch: {source['model_path']}")
    features = list(bundle.feature_cols)
    if len(features) != 132 or len(set(features)) != 132:
        raise RuntimeError("Expected 132 unique Magpie features")
    classes = list(bundle.pipeline.classes_)
    if len(classes) != 2 or set(classes) != {0, 1}:
        raise RuntimeError(f"Unexpected classifier classes: {classes}")
    threshold = 0.5 if bundle.threshold is None else float(bundle.threshold)
    if not np.isfinite(threshold) or not 0 <= threshold <= 1:
        raise RuntimeError(f"Invalid saved threshold: {threshold}")
    return {
        "run_id": bundle.run_id,
        "model_sha256": source["model_sha256"],
        "features_sha256": mlu.json_fingerprint(features),
        "mapping": mapping,
        "threshold": threshold,
        "features": features,
        "classes": classes,
        "spec": spec,
    }

def run_cross():
    logger = NotebookLogger(
        PROJECTS["COD"].PROJECT_DIR,
        prefix=f"{EXPERIMENT_ID}_cross",
        timestamp_fmt="%Y%m%d_%H%M%S_%f",
    )
    logger.start(header_lines=[
        f"Experiment: {EXPERIMENT_ID}",
        f"FORCE: {FORCE}",
        f"Protocol: {PROTOCOL_VERSION}; overlap: {OVERLAP_METHOD}; tolerance: {COMPOSITION_TOLERANCE}",
        f"Predictions: {SAVE_PREDICTIONS}; per element: {SAVE_PER_ELEMENT}; min_n: {ELEMENT_MIN_N}",
    ])
    started = time.perf_counter()
    state = None
    version_dir = None
    try:
        _require_sources()
        version_dir, state = _open_version()
        print(f"Version: {state['version']}")
        print(f"Output: {version_dir}")
        versions = {}
        for package in ("numpy", "pandas", "scikit-learn", "lightgbm", "pymatgen", "scipy", "joblib"):
            try:
                versions[package] = version(package)
            except PackageNotFoundError:
                versions[package] = "not installed"
        print("Versions:", versions)
        sources = [_load_source(item) for item in SOURCE_MODELS]
        fold_sha = {}
        for project, domain in TEST_TARGETS:
            folds = Path(PROJECTS[project].DATASET_FOLDS_TEMPLATE.format(domain=domain))
            fold_sha[(project, domain)] = mlu.file_sha256(folds)
        selected = []
        for source in sources:
            for target_project, target_domain in TEST_TARGETS:
                selected.append((source, target_project, target_domain))
        reusable, pending = [], []
        bundle_cache = {}
        for source, target_project, target_domain in selected:
            key = _task_key(source, target_project, target_domain)
            identity = {
                "task_key": key,
                "run_id": source["run_id"],
                "model_sha256": source["model_sha256"],
                "source_domain": source["domain"],
                "target_domain": target_domain,
                "source_folds_sha256": source["folds_sha256"],
                "target_folds_sha256": fold_sha[(target_project, target_domain)],
                "manifest_sha256": source["manifest_sha256"],
            }
            record = state["tasks"].get(key)
            if record and record.get("status") == "complete":
                loaded = bundle_cache.get(source["run_id"])
                if loaded is None:
                    loaded = _bundle_identity(mlu.load_bundle(source["model_path"]), source)
                    bundle_cache[source["run_id"]] = loaded
                identity.update({
                    "features_sha256": loaded["features_sha256"],
                    "mapping": loaded["mapping"],
                    "threshold": loaded["threshold"],
                })
                decision = _assess_saved(record, identity, version_dir)
            else:
                decision = "redo"
            (reusable if decision == "reuse" else pending).append(
                (source, target_project, target_domain, key)
            )
        print(
            f"Tasks completed={len(reusable)} skipped={len(reusable)} "
            f"remaining={len(pending)} selected={len(selected)}"
        )
        if not pending:
            print("All selected comparisons are complete. Reading saved results.")
            if state.get("status") != "complete":
                state["status"] = "complete"
                _save_state(version_dir, state)
            metrics = pd.DataFrame([
                row for record in state["tasks"].values()
                if record.get("status") == "complete" for row in record["metrics"]
            ])
            print("Results:", version_dir)
            return metrics, version_dir

        needed = {(item[0]["project"], item[0]["domain"]) for item in pending}
        needed.update((item[1], item[2]) for item in pending)
        metadata, tests = {}, {}
        for project, domain in tqdm(sorted(needed), desc="Read and validate folds", file=sys.stderr):
            key = f"{project}_{domain}"
            meta, test = _load_fold_data(PROJECTS[project], domain)
            meta["filepath"] = key + "|" + meta["id"]
            test["filepath"] = key + "|" + test["id"]
            metadata[key], tests[key] = meta, test
        combined = pd.concat(metadata.values(), ignore_index=True)
        if combined["filepath"].duplicated().any():
            raise RuntimeError("Duplicate cross-evaluation record identity")
        print("Joint composition grouping started.")
        with tqdm(total=1, desc="Joint composition groups", file=sys.stderr) as progress:
            grouped, bad = group_by_composition_and_fix_labels(
                combined[["filepath", "chem_formula_cif"]],
                combined[["filepath", "centrosymmetry"]],
                tolerance=COMPOSITION_TOLERANCE,
            )
            progress.update(1)
        if len(bad) or len(grouped) != len(combined):
            raise RuntimeError("Joint grouping lost records; cross evaluation stopped")
        group_map = grouped.set_index("filepath")["canonical_group_id"]
        train_groups = {}
        for key, meta in metadata.items():
            train_groups[key] = set(meta.loc[~meta.is_test, "filepath"].map(group_map))
            if key in tests:
                tests[key]["cross_group_id"] = tests[key]["filepath"].map(group_map)
                if tests[key]["cross_group_id"].isna().any():
                    raise RuntimeError(f"Missing joint composition key: {key}")
        _atomic_csv(
            version_dir / "cross_composition_groups.csv",
            grouped[["filepath", "canonical_exact_key", "canonical_group_id", "canonical_match_type"]],
        )
        del combined, grouped, bad, metadata, group_map

        loaded = {}
        with tqdm(total=len(selected), initial=len(reusable), desc="Cross model/test pairs",
                  unit="pair", file=sys.stderr) as progress:
            for source, target_project, target_domain, key in pending:
                pair_started = time.perf_counter()
                source_key = f"{source['project']}_{source['domain']}"
                target_key = f"{target_project}_{target_domain}"
                print(f"[COMPARE start] {key}")
                state["tasks"][key] = {"task_key": key, "status": "running"}
                _save_state(version_dir, state)
                if source["run_id"] not in loaded:
                    bundle = mlu.load_bundle(source["model_path"])
                    loaded[source["run_id"]] = (bundle, _bundle_identity(bundle, source))
                bundle, identity = loaded[source["run_id"]]
                features = identity["features"]
                test = tests[target_key]
                available = {col for col in test if col.startswith("MagpieData ")}
                if set(features) != available:
                    raise RuntimeError(f"Model/test feature mismatch: {key}")
                labelled, y_col = mlu.apply_target_spec(test.copy(), identity["spec"])
                if len(labelled) != len(test):
                    raise RuntimeError(f"Target mapping discarded records: {key}")
                probability = bundle.pipeline.predict_proba(labelled[features])[:, identity["classes"].index(1)]
                if not np.isfinite(probability).all() or ((probability < 0) | (probability > 1)).any():
                    raise RuntimeError(f"Invalid probabilities: {key}")
                threshold = identity["threshold"]
                pred = labelled[["id", "chem_formula_cif", "elements_cif", "group_id", "cross_group_id"]].copy()
                pred["y_true"] = labelled[y_col].to_numpy()
                pred["y_prob"] = probability
                pred["y_pred"] = (probability >= threshold).astype(int)
                pred["source_train_overlap"] = pred["cross_group_id"].isin(train_groups[source_key])
                n_overlap = int(pred["source_train_overlap"].sum())
                overlap = {
                    "model": source["model"], "source": source_key, "target": target_key,
                    "n_test": len(pred), "n_overlap": n_overlap, "n_disjoint": len(pred) - n_overlap,
                }
                artifacts = []
                overlap_name = f"overlap_{key}.csv"
                _atomic_csv(
                    version_dir / overlap_name,
                    pred.loc[pred.source_train_overlap, ["id", "chem_formula_cif", "cross_group_id"]],
                )
                artifacts.append({"name": overlap_name, "sha256": mlu.file_sha256(version_dir / overlap_name)})
                if SAVE_PREDICTIONS:
                    pred_name = f"predictions_{key}.csv"
                    _atomic_csv(version_dir / pred_name, pred)
                    artifacts.append({"name": pred_name, "sha256": mlu.file_sha256(version_dir / pred_name)})
                metric_rows = []
                for protocol, subset in (
                        ("all_test", pred),
                        ("composition_disjoint_test", pred.loc[~pred.source_train_overlap])):
                    metrics = (
                        mlu.compute_metrics(subset.y_true.to_numpy(), subset.y_prob.to_numpy(), threshold)
                        if len(subset) else
                        {"n": 0, **{name: np.nan for name in ("auc", "auprc", "bacc", "mcc", "acc", "f1")}}
                    )
                    result = {
                        "model": source["model"], "source": source_key, "target": target_key,
                        "run_id": identity["run_id"], "protocol": protocol, "threshold": threshold,
                        "n_original_test": len(pred), "n_source_train_overlap": n_overlap,
                        "status": "ok" if len(subset) else "empty_after_exclusion", **metrics,
                    }
                    metric_rows.append(result)
                    print("Cross metrics:", json.dumps(result, default=str))
                    if SAVE_PER_ELEMENT and len(subset):
                        element_name = f"elements_{key}_{protocol}.csv"
                        _atomic_csv(
                            version_dir / element_name,
                            mlu.report_by_element(
                                subset, y_true_col="y_true", min_n=ELEMENT_MIN_N, threshold=threshold,
                            ),
                        )
                        artifacts.append({"name": element_name, "sha256": mlu.file_sha256(version_dir / element_name)})
                duration = time.perf_counter() - pair_started
                for row in metric_rows:
                    row["pair_elapsed_seconds"] = duration
                task_identity = {
                    "task_key": key,
                    "run_id": identity["run_id"],
                    "model_sha256": source["model_sha256"],
                    "source_domain": source["domain"],
                    "target_domain": target_domain,
                    "source_folds_sha256": source["folds_sha256"],
                    "target_folds_sha256": fold_sha[(target_project, target_domain)],
                    "manifest_sha256": source["manifest_sha256"],
                    "features_sha256": identity["features_sha256"],
                    "mapping": identity["mapping"],
                    "threshold": threshold,
                }
                state["tasks"][key] = {
                    **task_identity,
                    "status": "complete",
                    "fingerprint": _task_fingerprint(task_identity),
                    "artifacts": artifacts,
                    "metrics": metric_rows,
                    "overlap": overlap,
                    "elapsed_seconds": duration,
                }
                _write_tables(version_dir, state)
                _save_state(version_dir, state)
                print(
                    f"[COMPARE saved] {key} n_test={len(pred)} n_overlap={n_overlap} "
                    f"elapsed_seconds={duration:.3f} state={version_dir / 'state.json'}"
                )
                progress.update(1)
        state["status"] = "complete"
        _save_state(version_dir, state)
        print("Cross evaluation complete:", version_dir)
        return pd.read_csv(version_dir / "cross_metrics.csv"), version_dir
    except BaseException:
        traceback.print_exc()
        if state is not None and version_dir is not None:
            state["status"] = "interrupted"
            _save_state(version_dir, state)
        raise
    finally:
        print(f"Cross elapsed_seconds: {time.perf_counter() - started:.3f}")
        logger.stop()


## Run the configured evaluation
Execute after reviewing the configuration above. With `FORCE = False`, completed compatible tasks are reused.


In [ ]:
cross_metrics, cross_output = run_cross()
cross_metrics
